# Redline: did Hermes 3 get less safe than Llama 3.1?

Runs the full before/after check from the README on Colab's free GPU. Nothing runs on your laptop.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Optional, for an AI grader instead of keyword rules: add a free Groq key in **🔑 Secrets** (left sidebar) as `GROQ_API_KEY`, with notebook access switched on. Get one at https://console.groq.com/keys.

About 20–40 minutes for 40 tests. Free hosted graders see the prompts: public test cases only, never private data.

In [ ]:
# 1. Settings
BEFORE = "ollama/llama3.1:8b"   # the base model
AFTER  = "ollama/hermes3:8b"    # a fine-tune of it
CASES  = 40
GRADER = "groq/openai/gpt-oss-120b"   # used only if the GROQ_API_KEY secret is set

In [ ]:
# 2. Install Redline and Ollama, start Ollama on the GPU, download both models
!pip -q install "git+https://github.com/kush0o7/Redline-Agent-Safety-Eval#subdirectory=backend"
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
for m in (BEFORE, AFTER):
    !ollama pull {m.split("/", 1)[1]}

In [ ]:
# 3. Run the comparison
import os
judge = "keyword"
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    judge = GRADER
    print("Grader:", GRADER)
except Exception:
    print("No GROQ_API_KEY secret: grading with keyword rules (less reliable).")
!redline compare {BEFORE} {AFTER} --cases {CASES} --packs core,financial_services --judge {judge} --out redline-out

In [ ]:
# 4. Download the report
from google.colab import files
files.download("redline-out/report.html")